# Scalar Shard Server Scatter Plot

이 notebook은 실행 중인 scalar shard 조회 서버(`serve_feature_query_api.py`)에서 feature 값을 가져오고, shard manifest에 연결된 `sample_meta.parquet`의 Y 값과 합쳐 `X vs Y` scatter plot을 Plotly로 그립니다.

기본 서버 주소는 `http://127.0.0.1:8000`입니다. 다른 포트로 서버를 띄웠다면 아래 설정 셀의 `SERVER_URL`만 바꾸면 됩니다.

In [ ]:
from pathlib import Path
import json

import polars as pl
import plotly.express as px
import requests
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "packages").exists() and (path / "python").exists():
            return path
    raise RuntimeError("repository root를 찾지 못했습니다. notebook을 repo 내부에서 실행해 주세요.")


REPO_ROOT = find_repo_root(Path.cwd())
SERVER_URL = "http://127.0.0.1:8000"

# 직접 지정하려면 이 값을 바꾸십시오.
MANIFEST_PATH = None

# Y column과 X feature 선택입니다. FEATURE_ID/FEATURE_KEY를 둘 다 None으로 두면
# /scalar/top-features에서 Y_COL 기준 top feature 하나를 자동 선택합니다.
Y_COL = "y"
FEATURE_ID = None
FEATURE_KEY = None

# 산점도 표시용 sample 수 제한입니다. 너무 큰 shard에서 브라우저 렌더링이 무거워지는 것을 막습니다.
MAX_SAMPLES = 5000


In [ ]:
def auto_manifest_path() -> Path:
    preferred = REPO_ROOT / "data" / "tmp_py_scalar_api_test" / "scalar_shard" / "scalar_shard_manifest.json"
    if preferred.exists():
        return preferred

    candidates = sorted(
        (REPO_ROOT / "data").glob("**/scalar_shard_manifest.json"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError("data/ 아래에서 scalar_shard_manifest.json을 찾지 못했습니다.")
    return candidates[0]


manifest_path = Path(MANIFEST_PATH).expanduser().resolve() if MANIFEST_PATH else auto_manifest_path()
print("manifest_path =", manifest_path)

health = requests.get(f"{SERVER_URL}/healthz", timeout=5)
health.raise_for_status()
print("server health =", health.json())


def post_json(path: str, payload: dict, *, timeout: int = 60) -> dict:
    response = requests.post(f"{SERVER_URL}{path}", json=payload, timeout=timeout)
    if not response.ok:
        raise RuntimeError(f"{path} failed: {response.status_code} {response.text}")
    return response.json()


schema = post_json("/scalar/schema", {"manifest_path": str(manifest_path)})
display(schema)


In [ ]:
def choose_feature() -> tuple[int | None, str | None]:
    if FEATURE_ID is not None and FEATURE_KEY is not None:
        raise ValueError("FEATURE_ID와 FEATURE_KEY는 둘 중 하나만 지정하십시오.")
    if FEATURE_ID is not None:
        return int(FEATURE_ID), None
    if FEATURE_KEY is not None:
        return None, str(FEATURE_KEY)

    try:
        top = post_json(
            "/scalar/top-features",
            {"manifest_path": str(manifest_path), "y_col": Y_COL, "top_k": 1},
        )
        if top.get("features"):
            item = top["features"][0]
            return int(item["feature_id"]), item.get("feature_key")
    except Exception as exc:
        print("top feature 자동 선택 실패. feature_id=0으로 fallback합니다:", exc)
    return 0, None


feature_id, feature_key = choose_feature()
feature_selector = {"feature_keys": [feature_key]} if feature_id is None else {"feature_ids": [feature_id]}
sample_ids = list(range(min(int(schema["n_samples"]), int(MAX_SAMPLES))))

payload = {
    "manifest_path": str(manifest_path),
    "sample_ids": sample_ids,
    "max_cells": len(sample_ids),
    **feature_selector,
}
feature_response = post_json("/scalar/features", payload)
feature = feature_response["features"][0]
feature_label = feature.get("feature_key") or f"feature_id={feature['feature_id']}"

x_df = pl.DataFrame(
    [
        {
            "sample_id": int(item["sample_id"]),
            "sample_key": item.get("sample_key"),
            "present": bool(item["present"]),
            "x": item.get("value"),
        }
        for item in feature["values"]
    ]
)

display({"selected_feature_id": feature["feature_id"], "selected_feature_key": feature.get("feature_key"), "sample_count": len(sample_ids)})
display(x_df.head())


In [ ]:
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))


def resolve_manifest_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path.resolve()
    return (manifest_path.parent / path).resolve()


sample_meta_path = resolve_manifest_path(manifest["sample_meta_path"])
sample_id_col = manifest.get("sample_id_col", "sample_id")
sample_key_col = manifest.get("sample_key_col", "sample_key")

sample_meta = pl.read_parquet(sample_meta_path)
if sample_id_col not in sample_meta.columns:
    sample_meta = sample_meta.with_row_index(sample_id_col)
if Y_COL not in sample_meta.columns:
    raise ValueError(f"sample_meta에 Y column이 없습니다: {Y_COL}")

select_cols = [sample_id_col, Y_COL]
if sample_key_col in sample_meta.columns:
    select_cols.append(sample_key_col)

y_df = sample_meta.select(select_cols).rename({sample_id_col: "sample_id"})
if sample_key_col in y_df.columns and sample_key_col != "sample_key":
    y_df = y_df.rename({sample_key_col: "sample_key_from_meta"})

plot_df = (
    x_df.join(y_df, on="sample_id", how="left")
    .filter(pl.col("present") & pl.col("x").is_not_null() & pl.col(Y_COL).is_not_null())
    .sort("sample_id")
)

print("sample_meta_path =", sample_meta_path)
print("plot rows =", plot_df.height)
display(plot_df.head())


In [ ]:
pdf = plot_df.to_pandas()
hover_cols = [col for col in ["sample_id", "sample_key"] if col in pdf.columns]

fig = px.scatter(
    pdf,
    x="x",
    y=Y_COL,
    hover_data=hover_cols,
    title=f"{feature_label}: X vs {Y_COL}",
    labels={"x": f"X = {feature_label}", Y_COL: f"Y = {Y_COL}"},
)
fig.update_traces(marker={"size": 7, "opacity": 0.75})
fig.update_layout(template="plotly_white", width=900, height=600)
fig.show()
